# 부채 자산 입력

## 0. 초기화

In [1]:
import ledgerly.utils as utils
from ledgerly.debt import (
    load_and_preprocess_debt_account,
    load_and_preprocess_debt_snapshot,
    upsert_debt_accounts,
    insert_debt_snapshots,
    fetch_current_debt_status,
    generate_debt_report
)

In [2]:
account_path = utils.find_project_root() / 'data' / 'raw' / '2603'/ 'dept_account.csv'
snapshot_path = utils.find_project_root() / 'data' / 'raw' /'2603'/  'dept_snapshot.csv'

## 1. 데이터 로드 및 전처리

In [3]:
account_df = load_and_preprocess_debt_account(account_path)
snapshot_df = load_and_preprocess_debt_snapshot(snapshot_path)

In [4]:
account_df.head()

,debt_id,owner,debt_name,initial_principal,repayment_type,maturity_date
0,학자금_1,주진영,학자금 대출,1115127,None,None
1,학자금_2,주진영,학자금 대출,1500000,None,None
2,학자금_3,주진영,학자금 대출,1500000,None,None
3,주택담보대출,임준열,주택담보대출,350000000,원리금 균등,2064-12-31
4,부모님_대출,임준열,부모님 대출,30000000,원금 균등,2035-12-31


In [5]:
snapshot_df.head()

,debt_id,snapshot_date,accrued_interest,interest_rate,principal_amount
0,학자금_1,2026-03-21,103,3.90,1115127
1,학자금_2,2026-03-21,170539,2.90,1670539
2,학자금_3,2026-03-21,170539,2.90,1670539
3,주택담보대출,2026-03-21,0,3.41,347116000
4,부모님_대출,2026-03-21,0,2.00,30000000


## 2. DB 저장 (Upsert)

In [6]:
# 계정 정보 업데이트
upsert_debt_accounts(account_df)

In [7]:
# 스냅샷 정보 저장
insert_debt_snapshots(snapshot_df)

## 3. 부채 현황 리포트

In [8]:
current_debt_status_df = fetch_current_debt_status()
report_df = generate_debt_report(current_debt_status_df)

In [9]:
report_df

,부채ID,소유주,부채 이름,최초 원금,남은 원금,이자율(%),누적 이자,정산 날짜
0,부모님_대출,임준열,부모님 대출,30000000,30000000,2.0,0,2026-03-21
1,주택담보대출,임준열,주택담보대출,350000000,347116000,3.41,0,2026-03-21
2,학자금_1,주진영,학자금 대출,1115127,1115127,3.9,103,2026-03-21
3,학자금_2,주진영,학자금 대출,1500000,1670539,2.9,170539,2026-03-21
4,학자금_3,주진영,학자금 대출,1500000,1670539,2.9,170539,2026-03-21
5,,,총계,,381572205,,,
